In [2]:
import pandas as pd

In [3]:
# Load the datasets
first_touch = pd.read_csv('first_touch.csv')
linear_attribution = pd.read_csv('linear_attribution.csv')
journey_master = pd.read_csv('journey_master.csv')

In [4]:
#Calculate the number of  campaigns and users in each dataset
print(journey_master["campaign_id"].nunique())
print(first_touch["user_id"].nunique())
print(linear_attribution["user_id"].nunique())

6
1000
1000


In [5]:
# Calculate conversion rate and conversions by channel
total_customers = first_touch['user_id'].nunique()
converted = first_touch[first_touch['revenue'].notna()]['user_id'].nunique()
conversion_rate = (converted / total_customers) * 100
conversions_by_channel = first_touch[first_touch['revenue'].notna()].groupby('channel')['user_id'].nunique()

print(conversions_by_channel)
print(f"Total customers: {total_customers}")
print(f"Converted customers: {converted}")
print(f"Conversion rate: {conversion_rate:.2f}%")

channel
Facebook      77
Google Ads    55
LinkedIn      60
TikTok        58
Name: user_id, dtype: int64
Total customers: 1000
Converted customers: 250
Conversion rate: 25.00%


In [ ]:
spend_by_user_channel = journey_master.groupby(['user_id', 'channel'])['spend'].sum().reset_index()
spend_by_user_channel.columns = ['user_id', 'channel', 'spend']

# Merge spend into linear_attribution
linear_attribution_with_spend = linear_attribution.merge(
spend_by_user_channel,
on=['user_id', 'channel'],
how='left'
)

first_touch_with_spend = first_touch.merge(
spend_by_user_channel,
on=['user_id', 'channel'],
how='left'
)

# Save the updated files
linear_attribution_with_spend.to_csv('linear_attribution.csv', index=False)
first_touch_with_spend.to_csv('first_touch.csv', index=False)

print("\nLinear Attribution preview:")
print(linear_attribution_with_spend.head())
print("\nFirst Touch preview:")
print(first_touch_with_spend.head())

In [6]:
TS_attribution = linear_attribution.groupby('channel')['spend'].sum().sort_values(ascending=False)
TS_firsttouch = first_touch.groupby('channel')['spend'].sum().sort_values(ascending=False)

print(TS_attribution)
print(TS_firsttouch)

channel
LinkedIn      214071232
TikTok        200396608
Facebook      187710680
Google Ads    186828402
Name: spend, dtype: int64
channel
LinkedIn      58543592
TikTok        51460542
Google Ads    50531735
Facebook      50234484
Name: spend, dtype: int64


In [7]:
TR_attribution = linear_attribution.groupby('channel')['revenue'].sum().sort_values(ascending=False)
TR_firsttouch = first_touch.groupby('channel')['revenue'].sum().sort_values(ascending=False)

print(TR_attribution)
print(TR_firsttouch)

channel
LinkedIn      123708.0
Google Ads    119721.0
Facebook      116264.0
TikTok        113502.0
Name: revenue, dtype: float64
channel
Facebook      34856.0
LinkedIn      34624.0
Google Ads    29460.0
TikTok        28292.0
Name: revenue, dtype: float64


In [8]:
# Calculate ROAS for both attribution models
ROAS_AT = (TR_attribution / TS_attribution)*100
print(ROAS_AT)

ROAS_FT = (TR_firsttouch / TS_firsttouch)*100
print(ROAS_FT)

channel
Facebook      0.061938
Google Ads    0.064081
LinkedIn      0.057788
TikTok        0.056639
dtype: float64
channel
Facebook      0.069387
Google Ads    0.058300
LinkedIn      0.059142
TikTok        0.054978
dtype: float64


In [9]:
conv_attribution = linear_attribution[linear_attribution['revenue'].notna()].groupby('channel')['user_id'].nunique()
conv_firsttouch = first_touch[first_touch['revenue'].notna()].groupby('channel')['user_id'].nunique()

spend_attribution = linear_attribution.groupby('channel')['spend'].sum()
spend_firsttouch = first_touch.groupby('channel')['spend'].sum()

cac_attribution = (spend_attribution / conv_attribution).fillna(0)
cac_firsttouch = (spend_firsttouch / conv_firsttouch).fillna(0)

print("CAC by channel - Linear Attribution")
print(cac_attribution)
print("CAC by channel - First Touch")
print(cac_firsttouch)

CAC by channel - Linear Attribution
channel
Facebook      1.124016e+06
Google Ads    1.160425e+06
LinkedIn      1.354881e+06
TikTok        1.301277e+06
dtype: float64
CAC by channel - First Touch
channel
Facebook      652395.896104
Google Ads    918758.818182
LinkedIn      975726.533333
TikTok        887250.724138
dtype: float64


In [10]:
# Linear attribution summary by channel
linear_summary = pd.DataFrame({
    'channel': TS_attribution.index,
    'spend': TS_attribution.values,
    'revenue': TR_attribution.reindex(TS_attribution.index).values,
    'roas_percent': ROAS_AT.reindex(TS_attribution.index).values
}).sort_values('roas_percent', ascending=False)

print(linear_summary)

      channel      spend   revenue  roas_percent
3  Google Ads  186828402  119721.0      0.064081
2    Facebook  187710680  116264.0      0.061938
0    LinkedIn  214071232  123708.0      0.057788
1      TikTok  200396608  113502.0      0.056639


In [11]:
first_touch_summary = pd.DataFrame({
    'channel': TS_firsttouch.index,
    'spend': TS_firsttouch.values,
    'revenue': TR_firsttouch.reindex(TS_firsttouch.index).values,
    'roas_percent': ROAS_FT.reindex(TS_firsttouch.index).values
}).sort_values('roas_percent', ascending=False)

print(first_touch_summary)

      channel     spend  revenue  roas_percent
3    Facebook  50234484  34856.0      0.069387
0    LinkedIn  58543592  34624.0      0.059142
2  Google Ads  50531735  29460.0      0.058300
1      TikTok  51460542  28292.0      0.054978


In [ ]:
linear_summary.to_csv('linear_summary.csv', index=False)
first_touch_summary.to_csv('first_touch_summary.csv', index=False)